# Batch, Async, and Fantasization Workflow

Batch BO、asynchronous BO、model fantasizationは関連しますが同じ機構ではありません。

```text
completed observations → model
                           ↓
unresolved evaluations → X_pending → acquisition → q candidates

model + fantasy sampler → fantasize() → fantasy-conditioned model batch
```

このNotebookではBoTorch-native semanticsをrobotorchan modelで実行し、各shapeと責務を確認します。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Completed observations define the fitted model

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.2 * X


train_X = torch.linspace(0.05, 0.95, 9).unsqueeze(-1)
train_Y = response(train_X)
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()
bounds = torch.tensor([[0.0], [1.0]])

## 3. Joint batch BO

`q=2` acquisitionは2候補をjoint decisionとして評価します。

In [ ]:
batch_acq = qLogNoisyExpectedImprovement(
    model=model,
    X_baseline=train_X,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=101),
)
batch_candidate, batch_value = optimize_acqf(
    acq_function=batch_acq,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=64,
)

assert batch_candidate.shape == torch.Size([2, 1])
print("joint q=2 candidate:", batch_candidate.squeeze(-1))
print("acquisition value:", batch_value)

## 4. Async BO uses X_pending as acquisition context

未完了の実験はtraining observationへ追加しません。`X_pending`としてacquisitionへ渡し、次候補がpending evaluationsを考慮するようにします。

In [ ]:
X_pending = torch.tensor([[0.32], [0.68]])
async_acq = qLogNoisyExpectedImprovement(
    model=model,
    X_baseline=train_X,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=202),
    X_pending=X_pending,
)
async_candidate, async_value = optimize_acqf(
    acq_function=async_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)

assert async_candidate.shape == torch.Size([1, 1])
print("pending points:", X_pending.squeeze(-1))
print("next async candidate:", async_candidate.squeeze(-1))
print("async acquisition value:", async_value)

## 5. Pending points are not training observations

pending pointにはまだ`Y`がありません。したがって`train_X/train_Y`へ追加してrefitするのではなく、acquisition contextとして保持します。

In [ ]:
assert model.train_inputs[0].shape[0] == train_X.shape[0]
assert async_acq.X_pending.shape == X_pending.shape
print("training rows:", train_X.shape[0])
print("pending rows:", async_acq.X_pending.shape[0])

## 6. Model fantasization is a different operation

`fantasize()`はpending contextそのものではなく、指定したXで仮想Yをposteriorからsampleし、その仮想観測で条件付けしたmodel batchを構築します。

In [ ]:
fantasy_X = torch.tensor([[0.40]])
fantasy_sampler = SobolQMCNormalSampler(torch.Size([4]), seed=303)
fantasy_model = model.fantasize(
    X=fantasy_X,
    sampler=fantasy_sampler,
)

test_X = torch.tensor([[0.20], [0.80]])
with torch.no_grad():
    base_mean = model.posterior(test_X).mean
    fantasy_mean = fantasy_model.posterior(test_X).mean

print("base posterior mean shape:", base_mean.shape)
print("fantasy posterior mean shape:", fantasy_mean.shape)

## 7. Fantasy batch dimension is not q

fantasy sample数4はoptimizerの`q=4`を意味しません。fantasy-conditioned modelのbatch dimensionとcandidate q dimensionは独立です。

In [ ]:
assert fantasy_mean.shape[-2:] == torch.Size([2, 1])
assert fantasy_mean.shape[0] == 4
print("fantasy batch size:", fantasy_mean.shape[0])
print("posterior q at test_X:", fantasy_mean.shape[-2])

## 8. Completed pending evaluation becomes training data

実験が完了して`Y`が得られた時点で初めてtraining dataへ追加し、通常のmodel update/refitへ戻ります。

In [ ]:
completed_X = X_pending[:1]
completed_Y = response(completed_X)
updated_X = torch.cat([train_X, completed_X], dim=0)
updated_Y = torch.cat([train_Y, completed_Y], dim=0)

updated_model = SingleTaskGP(updated_X, updated_Y)
fit_gpytorch_mll(updated_model.make_mll())
updated_model.eval()

remaining_pending = X_pending[1:]
updated_async_acq = qLogNoisyExpectedImprovement(
    model=updated_model,
    X_baseline=updated_X,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=404),
    X_pending=remaining_pending,
)
next_candidate, _ = optimize_acqf(
    acq_function=updated_async_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)

print("completed X:", completed_X.squeeze(-1))
print("remaining pending:", remaining_pending.squeeze(-1))
print("next candidate:", next_candidate.squeeze(-1))

## 9. Four independent axes

| Axis | Meaning |
| --- | --- |
| external `q` | real candidates selected in this optimization call |
| `X_pending` rows | real evaluations already launched but unresolved |
| model fantasy batch | hypothetical observations used for conditioning |
| one-shot auxiliary rows | acquisition-internal decision variables for KG/lookahead |

これらをshape上で相互に読み替えません。

## 10. Compatibility boundaries

- standard exact GPはq-batch、pending-aware acquisition、`fantasize()`を利用できます。
- MultiTask fantasy inputsではtask identityを正しく指定する必要があります。
- Kronecker MultiTask GPは現在`supports_fantasize=False`です。private GPyTorch cache workaroundは採用しません。
- empirical tree ensembleやNGBoostへExactGP fantasy semanticsを強制しません。
- multi-fidelity pending pointのfidelityは実際に走っているevaluation contextです。
- qMFKG/qKG内部fantasyはreal unresolved `X_pending`とは別です。
- mixed one-shot qKG/qMFKGにはaugmented batch全体のcategorical assignment semanticsがあります。

## 11. No custom async scheduler

robotorchanはscheduler stateを別抽象化しません。async BOは、completed observations、unresolved `X_pending`、pending-aware acquisition、candidate optimizerのcompositionとして表現します。

## 12. Related documentation

- `docs/optimization/batch-async-fantasization.md`
- `docs/optimization/batch-async-fantasization-final-audit.md`
- `docs/theory/acquisition/04_batch_noisy.md`
- `docs/theory/optimization/08_batch_one_shot.md`
- `31_acquisition_optimization.ipynb`

Batch、Async、Fantasizationは同じ「複数点」に見えても、real candidate、pending context、hypothetical conditioningという異なる意味を持ちます。